# Computing q, k, v for ONE word

In [1]:
import numpy as np 
d=3
corpus = "the cat sat on the mat".split()
vocab = list(set(corpus))
w2i = {w: i for i, w in enumerate(vocab)}
i2w = {i: w for i, w in enumerate(vocab)}

E = np.random.randn(len(vocab), d)

In [2]:
from scipy.special import softmax
d = 3 # dimension of my embedding for each word
dk = 3 # dimension of q/k/v - can differ from d, my choise

# The projection matrices are literally what changes the size — d is "before," dk is "after"
np.random.seed(42)
W_Q = np.random.randn(dk, d)
W_K = np.random.randn(dk, d)
W_V = np.random.randn(dk, d)

# test random x embedding to act as my input word embedding
x_1 = np.random.randn(d)

def project_qkv(x, W_Q, W_K, W_V):
    q = W_Q @ x 
    k = W_K @ x
    v = W_V @ x
    return q, k ,v

#q, k, v = project_qkv(x_1, W_Q, W_K, W_V)
#print(q, k ,v)


def project_sentence(words, E, w2i, W_Q, W_K, W_V):
    store = []
    for word in words:
        word_emb = E[w2i[word]]
        q, k, v = project_qkv(word_emb, W_Q, W_K, W_V)
        store.append([q, k, v])
    return store

store = project_sentence(corpus, E, w2i, W_Q, W_K, W_V)
#print(store)


def self_attention_scores(q_words, all_k):
    store_1 = []
    for q in q_words:
        store_2 = []
        for k in all_k:
            e_ij = np.dot(q, k)
            store_2.append(e_ij)
        store_1.append(store_2)
    return store_1 # each row represents each word q_i dot k_j score for all k in my corpus

all_q = [entry[0] for entry in store]
all_k = [entry[1] for entry in store]
all_v = [entry[2] for entry in store]

scores = self_attention_scores(all_q, all_k)
print(scores)


# lecture 5 slide 23 
def scale_scores(scores, dk):
    return np.array(scores)/np.sqrt(dk)

# helper functions used in parrallel with RNN
def attention_weights(scores):
    return softmax(scores)

def context_vector(weights, encoder_states):
    context = []
    for w, h in zip(weights, encoder_states):
        context.append(w * h)
    return sum(context)

# v = the value vector for one word (from project_qkv's third output)
def self_attention_output(scores, all_v, dk): 
    output = []
    for word_score in scores:
        scaled_score = scale_scores(word_score, dk)
        context_score = context_vector(attention_weights(scaled_score), all_v)
        output.append(context_score)
    return output

attention_output = self_attention_output(scores, all_v, dk)
print(attention_output)

[[np.float64(-1.2665911642814653), np.float64(2.587427533690588), np.float64(0.7799201849347374), np.float64(0.22757764072163206), np.float64(-1.2665911642814653), np.float64(0.600085082758379)], [np.float64(-6.29602546241401), np.float64(8.094340737588688), np.float64(6.0603329339938306), np.float64(-1.7793673966791306), np.float64(-6.29602546241401), np.float64(1.6517859671752493)], [np.float64(-1.7274244555479055), np.float64(2.0294573541192302), np.float64(1.4112028033596478), np.float64(-0.377038871816691), np.float64(-1.7274244555479055), np.float64(0.09337675870756898)], [np.float64(-0.7759431429932733), np.float64(1.149945878111112), np.float64(0.9051050609319229), np.float64(-0.27951648852922606), np.float64(-0.7759431429932733), np.float64(0.45205647057842474)], [np.float64(-1.2665911642814653), np.float64(2.587427533690588), np.float64(0.7799201849347374), np.float64(0.22757764072163206), np.float64(-1.2665911642814653), np.float64(0.600085082758379)], [np.float64(-7.8201758

# Feed-forwrad network

In [3]:
hidden_size = 4
W1 = np.random.randn(hidden_size, dk)
b1 = np.random.randn(hidden_size)
W2 = np.random.randn(dk, hidden_size)
b2 = np.random.randn(dk)

def relu(x):
    return np.maximum(x, 0)

#residual and layer_norm both operate on one word's vector at a time — same as feed_forward
def feed_forward(x, W1, b1, W2, b2):
    # x: ONE word's attention_output, shape (3,) -- applied PER WORD,
    # independently -- no cross-word mixing happens here (that already
    # happened in self-attention)
    inner = W1 @ x + b1     # (3,) -> (4,)
    layer1 = relu(inner)    # (4,) still
    output = W2 @ layer1 + b2  # (4,) -> (3,) -- back to original dk size
    return output

def residual(input_x, output_x):
    # x_l = F(x_l-1) + x_l-1 -- adds a layer's OWN input back onto its
    # OWN output. input_x and output_x MUST be the same shape.
    # CAUTION: attention_output[i] is (3,)=dk, but the ORIGINAL embedding
    # x_i is (5,)=d. Since dk != d here, residual(x_i, attention_output[i])
    # will NOT work as-is -- real Transformers set dk == d specifically
    # so every residual connection adds matching shapes. Fix: either set
    # dk = d, or only apply this residual to same-shaped pairs (e.g.
    # residual(attention_output[i], ffn_output[i]), which ARE both (3,))
    return input_x + output_x

def layer_norm(x, epsilon=1e-5):
    # normalizes ONE word's OWN vector (its own mean/std, not across
    # words, not across a batch) -- keeps values stable through many
    # stacked layers
    u = np.mean(x)
    sigma = np.std(x)
    return (x - u) / (sigma + epsilon)

word = corpus[0]


# Multi-head attention

In [19]:
# h = num_heads, d_h = d // num_heads (each head's smaller dimension)
# runs self-attention h SEPARATE times, each with its OWN random W_Q/W_K/W_V
# softmax happens INSIDE self_attention_output -> attention_weights -> softmax
# layers structure: layers[head_index][word_index] = that word's (d_h,) vector FOR that head
d = 6
d = 6
E_2 = np.random.randn(len(vocab), d)
def multi_head_attention(words, E, w2i, num_heads, d):
    layers = []
    for heads in range(num_heads):
        W_Q = np.random.randn(d//num_heads, d)
        W_K = np.random.randn(d//num_heads ,d)
        W_V = np.random.randn(d//num_heads,d)
        store = project_sentence(words, E, w2i, W_Q, W_K, W_V)
        all_q = [entry[0] for entry in store]
        all_k = [entry[1] for entry in store]
        all_v = [entry[2] for entry in store]
        scores = self_attention_scores(all_q, all_k)
        attention_output = self_attention_output(scores, all_v, d//num_heads)
        layers.append(attention_output)
    return layers

layers = multi_head_attention(corpus, E_2, w2i, 3, 6)
print(len(layers)==3)
print(len(layers[0])==6)
print(layers)

# COMBINE HEADS (slide 30: H = [H_1;...;H_h], MultiHead = W_O . H)
# for each word: gather its vector from ALL heads, concatenate back to (d,),
# then ONE final linear mix via W_O
W_O = np.random.randn(d, d)

def combine_heads(layers, W_O, num_words):
    outputs = []
    for i in range(num_words): # one word at a time
        H = []
        for head in layers:
            H.append(head[i]) # word i's (d_h,) vector from THIS head
        # H so far = [head1_vec, head2_vec, head3_vec] -- 3 SEPARATE pieces    
        H_concat = np.concatenate(H) # glue into ONE (d,) vector
        outputs.append(W_O @ H_concat) # one mix -> word i's final (d,) output   
    return outputs

outputs = combine_heads(layers, W_O, len(corpus))
print(len(outputs))
print(outputs)

True
True
[[array([0.6392013, 0.4586145]), array([-1.5722694, -0.6056999]), array([1.39238924, 1.35342773]), array([0.53248301, 0.43922148]), array([0.6392013, 0.4586145]), array([-2.68583847, -1.89735238])], [array([-1.85260425, -1.78400556]), array([-1.15614016,  0.23742951]), array([-0.04074944,  1.7814464 ]), array([1.22110615, 2.99612672]), array([-1.85260425, -1.78400556]), array([-1.15789946,  0.23499472])], [array([0.66113743, 0.52966744]), array([-5.20695237,  1.28052012]), array([ 0.96631109, -2.07052331]), array([-6.15199176,  1.43384919]), array([0.66113743, 0.52966744]), array([ 1.53423348, -2.49193962])]]
6
[array([-0.98852144,  5.41332627,  6.62404596, -2.43303049, -0.25079305,
        1.66063436]), array([-1.99726537, -5.41609845, -7.012492  , -7.5526529 ,  7.22966098,
        6.83120739]), array([ 2.88613764, -1.10294926,  1.00789212,  3.56789274, -5.77018726,
       -3.13916621]), array([ -4.41140098, -14.19497496, -12.67500295,  -5.68808111,
         2.92391013,   5.